# Adaptive Moving Average and SES Forecasting Demo

This demo evaluates adaptive moving average models ($K \in \{1, 3, 5, 10\}$) and Simple Exponential Smoothing (SES) models ($\alpha \in \{0.2, 0.5, 0.8\}$) against standard naive persistence baselines on synthetic time series instances.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-0ece95-adaptive-smoothing-and-persistence-trade/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded dataset with {len(data['datasets'][0]['examples'])} examples.")

## Configuration
Define tunable parameters for demonstration scale (e.g. number of examples to process).

In [ ]:
# Configuration parameters
MAX_EXAMPLES = 3  # Minimum / demo scale

## Processing and Evaluation
Iterate through the time series examples, compute moving averages and simple exponential smoothing forecasts, and record predictions.

In [ ]:
dataset_list = data['datasets']
out_datasets = []

for ds in dataset_list:
    ds_name = ds.get('dataset', 'synthetic_time_series')
    examples = ds['examples'][:MAX_EXAMPLES]
    new_examples = []
    
    for item in examples:
        series = np.array(json.loads(item['input']) if isinstance(item['input'], str) else item['input'])
        
        history = series[:-1]
        actual = series[-1]
        
        f_naive = history[-1]
        
        def get_ma(k, hist):
            if len(hist) < k:
                return np.mean(hist)
            return np.mean(hist[-k:])
            
        f_ma1 = get_ma(1, history)
        f_ma3 = get_ma(3, history)
        f_ma5 = get_ma(5, history)
        f_ma10 = get_ma(10, history)
        
        def get_ses(alpha, hist):
            s = hist[0]
            for val in hist[1:]:
                s = alpha * val + (1 - alpha) * s
            return s
            
        f_ses02 = get_ses(0.2, history)
        f_ses05 = get_ses(0.5, history)
        f_ses08 = get_ses(0.8, history)
        
        ex_out = {
            "input": item['input'],
            "output": str(actual),
            "metadata_id": item.get('metadata_id', 0),
            "predict_naive": str(f_naive),
            "predict_ma_1": str(f_ma1),
            "predict_ma_3": str(f_ma3),
            "predict_ma_5": str(f_ma5),
            "predict_ma_10": str(f_ma10),
            "predict_ses_0.2": str(f_ses02),
            "predict_ses_0.5": str(f_ses05),
            "predict_ses_0.8": str(f_ses08)
        }
        new_examples.append(ex_out)
        
    out_datasets.append({
        "dataset": ds_name,
        "examples": new_examples
    })

output = {
    "status": "success",
    "datasets": out_datasets,
    "summary": "Evaluated moving averages and SES against naive baseline with per-example predictions."
}

print("Processing complete. Evaluated", len(out_datasets[0]['examples']), "examples.")

## Results Summary & Visualization
Display predicted values against actual targets and compute Mean Absolute Error (MAE) across models for the evaluated examples.

In [ ]:
models = ['predict_naive', 'predict_ma_1', 'predict_ma_3', 'predict_ma_5', 'predict_ma_10', 'predict_ses_0.2', 'predict_ses_0.5', 'predict_ses_0.8']
examples_processed = out_datasets[0]['examples']

mae_results = {m: [] for m in models}

for ex in examples_processed:
    actual_val = float(ex['output'])
    for m in models:
        pred_val = float(ex[m])
        mae_results[m].append(abs(actual_val - pred_val))

mean_mae = {m: np.mean(vals) for m, vals in mae_results.items()}

print("Mean Absolute Error (MAE) by Model:")
for m, mae in mean_mae.items():
    print(f"  {m:18s}: {mae:.4f}")

# Plotting model MAEs
plt.figure(figsize=(10, 5))
model_names = list(mean_mae.keys())
maes = list(mean_mae.values())
plt.bar(model_names, maes, color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Mean Absolute Error (MAE)")
plt.title("Forecasting Performance Comparison Across Models")
plt.tight_layout()
plt.show()